# Assignment 07 — EC: sentiment of e-commerce review comments with a 5-layer RNN and LSTM

**Dataset:** *Amazon Reviews: Unlocked Mobile Phones* (PromptCloud, Kaggle), 413,840 reviews of about 4,400 phones sold on Amazon, each with a 1–5 star rating and the review text (132 MB CSV).
**Task:** read the review text and predict whether the customer is satisfied: **positive** (4–5 stars) or **negative** (1–2 stars). 3-star reviews are ambiguous and left out.

**Models:** the same architecture four times — {5-layer RNN, 5-layer LSTM} × {Keras, PyTorch}:
word ids → embedding (64-d) → 5 recurrent layers (64 units) → last state → Dense(1) → sigmoid.
Run IDs: `EC-TF-RNN`, `EC-TF-LSTM`, `EC-PT-RNN`, `EC-PT-LSTM`. The structure of both networks is derived step by step in `00_rnn_lstm_concepts.ipynb`.

## 1. Setup

One seed (42) for Python, NumPy, TensorFlow and PyTorch. Every run uses the same data arrays, the same architecture, the same optimizer (Adam, learning rate $10^{-3}$, gradient clipping at norm 1), batch size 256, at most 10 epochs and early stopping (patience 2) on the validation loss, keeping the best epoch.

In [ ]:
import os, re, time, json, random, collections, warnings
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA = Path("../dataset/ec")
RES = Path("results"); RES.mkdir(parents=True, exist_ok=True)
IMG = Path("../report/images"); IMG.mkdir(parents=True, exist_ok=True)

VOCAB, MAXLEN, EMB, H, LAYERS = 20_000, 100, 64, 64, 5
BATCH, LR, CLIP = 256, 1e-3, 1.0
EPOCHS, PATIENCE = 10, 2
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
print("results ->", RES.resolve())

### Hyperparameters

| Name | Value | Meaning |
|---|---|---|
| vocabulary | 20,000 words | the most frequent words of the **training** reviews; others become `<unk>` |
| sequence length $T$ | 100 tokens | covers 93 % of reviews completely; longer ones are cut after 100 words |
| embedding | 64 | each word id becomes a learned 64-d vector |
| hidden units $H$ | 64 per layer | |
| layers $L$ | 5 | stacked: layer $l$ reads the states of layer $l-1$ |
| output | Dense(1) + sigmoid | $P(\text{positive})$ |
| loss | binary cross-entropy | $-[y\ln p+(1-y)\ln(1-p)]$ |
| optimizer | Adam, lr $10^{-3}$, clip-norm 1 | clipping limits exploding gradients |
| batch / epochs | 256 / ≤ 10, early stopping patience 2 | |

## 2. Data

Raw columns: product, brand, price, **Rating**, **Reviews** (text), review votes. Steps: drop empty reviews → **drop duplicates** (the same review is listed once per phone variant, so the same text appears many times; keeping duplicates would leak copies of a test review into training) → drop 3-star reviews → label $y=1$ for 4–5 stars, $y=0$ for 1–2 stars.

In [ ]:
raw = pd.read_csv(DATA / "Amazon_Unlocked_Mobile.csv")
print("raw shape:", raw.shape)
display(raw.head(3))
print("empty reviews:", raw.Reviews.isna().sum(), "| duplicated review texts:", raw.Reviews.duplicated().sum())

In [ ]:
df = raw.dropna(subset=["Reviews"]).drop_duplicates(subset=["Reviews"]).copy()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
raw.Rating.value_counts().sort_index().plot.bar(ax=ax[0], color="#999", rot=0, title="all rows (with duplicates)")
df.Rating.value_counts().sort_index().plot.bar(ax=ax[1], color="#2a78d6", rot=0, title=f"unique reviews ({len(df):,})")
for a in ax: a.set_xlabel("stars")
plt.tight_layout(); plt.show()
df = df[df.Rating != 3].copy()
df["y"] = (df.Rating >= 4).astype(int)
print(f"after removing 3-star reviews: {len(df):,} reviews, positive share {df.y.mean():.3f}")

**Reading the plot.** Most customers give 5 stars, and 1 star is the second most common rating: people write reviews when they are very happy or very unhappy. After de-duplication about 3 reviews in 4 are positive, so a model that always says "positive" is already right ~75 % of the time; accuracy alone is therefore not enough, and macro-F1 and ROC-AUC are reported too.

### Review length

A recurrent network reads one word per step, so the length of a review is the number of steps.

In [ ]:
df["n_words"] = df.Reviews.str.split().str.len()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].hist(df.n_words.clip(upper=300), bins=60, color="#2a78d6"); ax[0].axvline(MAXLEN, c="C3", ls="--", label=f"T = {MAXLEN}")
ax[0].set_xlabel("words per review (clipped at 300)"); ax[0].legend()
for y, name in [(1, "positive"), (0, "negative")]:
    ax[1].hist(df.n_words[df.y == y].clip(upper=300), bins=60, alpha=0.6, density=True, label=name)
ax[1].set_xlabel("words per review"); ax[1].legend(); plt.tight_layout(); plt.show()
print(df.groupby("y").n_words.describe()[["50%", "75%", "max"]].rename(index={0: "negative", 1: "positive"}))
print(f"share of reviews with <= {MAXLEN} words: {(df.n_words <= MAXLEN).mean():.3f}")

Negative reviews are longer: an unhappy customer explains what went wrong. So the evidence for a negative label is often spread over many words, a case where a model's memory matters.

### Which words separate the classes?

Before any neural network: the words with the largest frequency difference between negative and positive reviews.

In [ ]:
def tokenize(text):
    """lower-case, keep letters, digits and apostrophes: "Don't buy!!" -> ["don't", "buy"]"""
    return re.findall(r"[a-z0-9']+", text.lower())

pos_c, neg_c = collections.Counter(), collections.Counter()
for text, y in zip(df.Reviews.values[:60_000], df.y.values[:60_000]):
    (pos_c if y else neg_c).update(set(tokenize(text)))
n_pos, n_neg = (df.y.values[:60_000] == 1).sum(), (df.y.values[:60_000] == 0).sum()
words = [w for w, c in (pos_c + neg_c).most_common(3000)]
diff = pd.Series({w: neg_c[w] / n_neg - pos_c[w] / n_pos for w in words}).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
diff.head(15).plot.barh(ax=ax[0], color="#3aa06a", title="more frequent in positive reviews")
diff.tail(15).plot.barh(ax=ax[1], color="#d64545", title="more frequent in negative reviews")
plt.tight_layout(); plt.show()

## 3. From text to numbers, step by step

A network cannot read text. The review goes through four steps, shown on one real review:

1. **tokenize**: split into lower-case words;
2. **vocabulary**: give the 20,000 most frequent training words an id (1 = `<unk>` for unknown words, 0 = padding);
3. **encode**: replace each word by its id;
4. **pad / truncate** to $T=100$: shorter reviews get zeros **in front** (`pre` padding, so the last steps the network reads are real words and the final state $h_T$ has just seen the text); longer reviews keep their first 100 words.

The data are split **before** building the vocabulary (70 % train, 15 % validation, 15 % test, stratified by label), so the vocabulary never sees validation or test words.

In [ ]:
idx = np.arange(len(df))
i_tr, i_tmp = train_test_split(idx, test_size=0.30, stratify=df.y, random_state=SEED)
i_va, i_te = train_test_split(i_tmp, test_size=0.50, stratify=df.y.values[i_tmp], random_state=SEED)
texts, y_all = df.Reviews.values, df.y.values.astype("float32")
print(f"train {len(i_tr):,} | validation {len(i_va):,} | test {len(i_te):,}")

counts = collections.Counter(w for t in texts[i_tr] for w in tokenize(t))
itos = ["<pad>", "<unk>"] + [w for w, _ in counts.most_common(VOCAB - 2)]
stoi = {w: i for i, w in enumerate(itos)}
coverage = sum(counts[w] for w in itos[2:]) / sum(counts.values())
print(f"{len(counts):,} distinct training words; the top {VOCAB:,} cover {coverage:.1%} of all word occurrences")

def encode(text):
    ids = [stoi.get(w, 1) for w in tokenize(text)][:MAXLEN]          # truncate: keep the first T words
    return [0] * (MAXLEN - len(ids)) + ids                            # pre-pad with 0

ex = texts[i_tr][3]
print("\n1. review   :", ex[:160])
print("2. tokens   :", tokenize(ex)[:14])
print("3. ids      :", [stoi.get(w, 1) for w in tokenize(ex)][:14])
print("4. padded   :", encode(ex)[:6], "...", encode(ex)[-8:], f"(length {len(encode(ex))})")
print("   decoded  :", " ".join(itos[i] for i in encode(ex) if i)[:160])

In [ ]:
X_all = np.array([encode(t) for t in texts], dtype="int32")
Xtr, Xva, Xte = X_all[i_tr], X_all[i_va], X_all[i_te]
ytr, yva, yte = y_all[i_tr], y_all[i_va], y_all[i_te]
print("X_train:", Xtr.shape, "(reviews, T)  y_train:", ytr.shape, f"positive share {ytr.mean():.3f}")
print("share of <unk> among real tokens:", f"{(Xtr == 1).sum() / (Xtr > 0).sum():.3%}")

**Shapes through the network** for a batch of 256 reviews:
`(256, 100)` word ids → embedding → `(256, 100, 64)` → layer 1 … layer 5, each `(256, 100, 64)` → last step of layer 5 `(256, 64)` → Dense `(256, 1)` → sigmoid → $P(\text{positive})$.

### Shared evaluation helpers

Every run goes through `finalize`: predicted probabilities on the test set, threshold 0.5, then accuracy, macro-F1 (the mean of the F1 scores of both classes, so the small negative class counts as much as the large positive one), F1 of the negative class and ROC-AUC (the probability that a random positive review gets a higher score than a random negative one; 0.5 = guessing, 1 = perfect).

In [ ]:
RUNS = {}

def metrics(y, p):
    yhat = (p >= 0.5).astype(int)
    return dict(accuracy=float(accuracy_score(y, yhat)), f1_macro=float(f1_score(y, yhat, average="macro")),
                f1_negative=float(f1_score(y, yhat, pos_label=0)), roc_auc=float(roc_auc_score(y, p)))

def finalize(run_id, p_test, history, params, sec_epoch, total_s):
    rec = dict(run_id=run_id, params=int(params), sec_per_epoch=round(sec_epoch, 2), train_seconds=round(total_s, 1),
               epochs=len(history["loss"]), best_epoch=int(np.argmin(history["val_loss"]) + 1),
               history=history, **metrics(yte, p_test))
    RUNS[run_id] = {**rec, "p_test": p_test}
    json.dump(rec, open(RES / f"{run_id}.json", "w"), indent=1)
    print(f"{run_id}: acc {rec['accuracy']:.4f}  F1-macro {rec['f1_macro']:.4f}  AUC {rec['roc_auc']:.4f}  "
          f"epochs {rec['epochs']} (best {rec['best_epoch']})  {sec_epoch:.0f} s/epoch")
    return rec

def rnn_params(F, H, L, nb=1):  return F * H + H * H + nb * H + (L - 1) * (2 * H * H + nb * H)
def lstm_params(F, H, L, nb=1): return 4 * rnn_params(F, H, L, nb)
EXPECTED = {"RNN": VOCAB * EMB + rnn_params(EMB, H, LAYERS) + H + 1, "LSTM": VOCAB * EMB + lstm_params(EMB, H, LAYERS) + H + 1}
print("expected parameters (Keras-style, one bias):", {k: f"{v:,}" for k, v in EXPECTED.items()})

## 4. Baselines

- **Majority class**: always predict "positive".
- **TF-IDF + logistic regression**: a strong classical model that ignores word order: each review becomes a weighted bag of words and bigrams, and a linear model scores it. If a recurrent network cannot beat this, its ability to read order is not being used.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
BASE = {}
BASE["majority"] = metrics(yte, np.ones_like(yte) * 0.99)
t0 = time.time()
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000, sublinear_tf=True)
lr = LogisticRegression(max_iter=2000, C=4).fit(tfidf.fit_transform(texts[i_tr]), ytr)
p_lr = lr.predict_proba(tfidf.transform(texts[i_te]))[:, 1]
BASE["tfidf_logreg"] = metrics(yte, p_lr)
print(f"TF-IDF + logistic regression trained in {time.time() - t0:.0f} s")
display(pd.DataFrame(BASE).T.round(4))

## 5. Keras — runs `EC-TF-RNN`, `EC-TF-LSTM`

In Keras the 5-layer network is a list of layers: `Embedding` → four recurrent layers with `return_sequences=True` (they pass the whole sequence of states up) → a fifth with `return_sequences=False` (it returns only its last state $h^{(5)}_T$) → `Dense(1)`.
`SimpleRNN` computes $h_t=\tanh(x_tW+h_{t-1}U+b)$; `LSTM` adds the gates. Keras's defaults: Glorot-uniform input weights, orthogonal recurrent weights, zero biases, LSTM forget-gate bias 1. `model.fit` runs the whole training loop.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

def build_keras(kind):
    cell = {"RNN": layers.SimpleRNN, "LSTM": layers.LSTM}[kind]
    stack = [layers.Input((MAXLEN,), dtype="int32"), layers.Embedding(VOCAB, EMB)]
    stack += [cell(H, return_sequences=(l < LAYERS - 1)) for l in range(LAYERS)]
    stack += [layers.Dense(1, activation="sigmoid")]
    m = keras.Sequential(stack, name=f"keras_{kind}")
    m.compile(optimizer=keras.optimizers.Adam(LR, clipnorm=CLIP), loss="binary_crossentropy", metrics=["accuracy"])
    return m

class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.times = []
    def on_epoch_begin(self, epoch, logs=None): self.t = time.time()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.time() - self.t)

def run_keras(kind):
    tf.keras.utils.set_random_seed(SEED)
    m = build_keras(kind)
    assert m.count_params() == EXPECTED[kind], (m.count_params(), EXPECTED[kind])
    timer = EpochTimer(); t0 = time.time()
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=BATCH, verbose=2,
              callbacks=[keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True), timer])
    p = m.predict(Xte, batch_size=1024, verbose=0)[:, 0]
    hist = {"loss": h.history["loss"], "val_loss": h.history["val_loss"], "acc": h.history["accuracy"], "val_acc": h.history["val_accuracy"]}
    return finalize(f"EC-TF-{kind}", p, hist, m.count_params(), float(np.median(timer.times[1:] or timer.times)), time.time() - t0)

build_keras("RNN").summary()

The parameter count matches the formula from the concepts notebook: embedding $20{,}000\cdot64=1{,}280{,}000$, layer 1 $64\cdot64+64\cdot64+64=8{,}256$, layers 2–5 the same each, head $65$. Almost all parameters sit in the embedding; the 5 recurrent layers hold 41,280 (RNN) or 165,120 (LSTM).

In [ ]:
run_keras("RNN");

In [ ]:
run_keras("LSTM");

## 6. PyTorch — runs `EC-PT-RNN`, `EC-PT-LSTM`

PyTorch writes the same network as a class: `nn.Embedding` → `nn.RNN(64, 64, num_layers=5, batch_first=True)` (or `nn.LSTM`) → take the last step of the top layer → `nn.Linear(64, 1)`. One call to `nn.RNN(num_layers=5)` is the whole stack $f^{(5)}\circ\dots\circ f^{(1)}$. The training loop is written by hand: `zero_grad → forward → loss → backward → clip → step`.

To make both frameworks start alike, the PyTorch weights get Keras's initialisation (Glorot-uniform input weights, orthogonal recurrent weights, zero biases, LSTM forget bias 1, embedding uniform in ±0.05). PyTorch keeps **two** bias vectors per layer (`bias_ih`, `bias_hh`), so it has $5\cdot64=320$ (RNN) or $5\cdot256=1{,}280$ (LSTM) more parameters, all starting at zero; the sum of the two is Keras's single bias.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(SEED)
DEV = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEV)

class TorchSentiment(nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, EMB)
        self.rnn = {"RNN": nn.RNN, "LSTM": nn.LSTM}[kind](EMB, H, num_layers=LAYERS, batch_first=True)
        self.head = nn.Linear(H, 1)
        nn.init.uniform_(self.emb.weight, -0.05, 0.05)
        for name, p in self.rnn.named_parameters():                # Keras-style initialisation
            if name.startswith("weight_ih"):
                for blk in p.data.chunk(4 if kind == "LSTM" else 1): nn.init.xavier_uniform_(blk)
            elif name.startswith("weight_hh"):
                for blk in p.data.chunk(4 if kind == "LSTM" else 1): nn.init.orthogonal_(blk)
            else:
                nn.init.zeros_(p)
                if kind == "LSTM" and name.startswith("bias_ih"): p.data[H:2 * H] = 1.0   # forget gate
        nn.init.xavier_uniform_(self.head.weight); nn.init.zeros_(self.head.bias)
    def forward(self, x):                      # x: (B, T) word ids
        out, _ = self.rnn(self.emb(x))         # out: (B, T, H), states of the top layer
        return self.head(out[:, -1]).squeeze(1)   # logit from h^(5)_T

def predict_torch(model, X, bs=1024):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(torch.sigmoid(model(torch.from_numpy(X[i:i + bs]).long().to(DEV))).cpu().numpy())
    return np.concatenate(out)

def run_torch(kind):
    torch.manual_seed(SEED)
    m = TorchSentiment(kind).to(DEV)
    n = sum(p.numel() for p in m.parameters()); extra = LAYERS * H * (4 if kind == "LSTM" else 1)
    assert n == EXPECTED[kind] + extra, (n, EXPECTED[kind], extra)
    opt = torch.optim.Adam(m.parameters(), lr=LR, eps=1e-7); lossf = nn.BCEWithLogitsLoss()
    Xt, yt = torch.from_numpy(Xtr).long(), torch.from_numpy(ytr)
    Xv, yv = torch.from_numpy(Xva).long().to(DEV), torch.from_numpy(yva).to(DEV)
    hist = {"loss": [], "val_loss": [], "acc": [], "val_acc": []}; times = []; best, best_state, wait = np.inf, None, 0
    g = torch.Generator().manual_seed(SEED); t0 = time.time()
    for ep in range(EPOCHS):
        m.train(); te = time.time(); tot, correct = 0.0, 0
        for b in torch.randperm(len(Xt), generator=g).split(BATCH):
            xb, yb = Xt[b].to(DEV), yt[b].to(DEV)
            logit = m(xb); loss = lossf(logit, yb)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), CLIP); opt.step()
            tot += loss.item() * len(b); correct += ((logit > 0).float() == yb).sum().item()
        m.eval()
        with torch.no_grad():
            vl, vc = 0.0, 0
            for i in range(0, len(Xv), 1024):
                lo = m(Xv[i:i + 1024]); vl += lossf(lo, yv[i:i + 1024]).item() * len(lo); vc += ((lo > 0).float() == yv[i:i + 1024]).sum().item()
        times.append(time.time() - te)
        hist["loss"].append(tot / len(Xt)); hist["val_loss"].append(vl / len(Xv)); hist["acc"].append(correct / len(Xt)); hist["val_acc"].append(vc / len(Xv))
        print(f"epoch {ep + 1}: loss {hist['loss'][-1]:.4f}  val_loss {hist['val_loss'][-1]:.4f}  val_acc {hist['val_acc'][-1]:.4f}  {times[-1]:.0f}s")
        if hist["val_loss"][-1] < best - 1e-4:
            best, wait = hist["val_loss"][-1], 0; best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE: break
    m.load_state_dict(best_state)
    return finalize(f"EC-PT-{kind}", predict_torch(m, Xte), hist, n, float(np.median(times[1:] or times)), time.time() - t0)

print(TorchSentiment("LSTM"))

In [ ]:
run_torch("RNN");

In [ ]:
run_torch("LSTM");

## 7. Comparison

All four runs and the two baselines on the same test reviews.

In [ ]:
rows = [{"run": k, **{m: v[m] for m in ["accuracy", "f1_macro", "f1_negative", "roc_auc", "params", "sec_per_epoch", "epochs", "best_epoch"]}} for k, v in RUNS.items()]
rows += [{"run": f"baseline: {k}", **v} for k, v in BASE.items()]
table = pd.DataFrame(rows).set_index("run")
table.to_csv(RES / "EC_comparison.csv")
display(table.round(4))

### Learning curves

Training and validation loss per epoch. A validation loss that rises while training loss falls means overfitting; early stopping keeps the epoch with the lowest validation loss.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for k, v in RUNS.items():
    ls = "-" if "-TF-" in k else "--"; c = "C0" if k.endswith("RNN") else "C3"
    ax[0].plot(range(1, len(v["history"]["val_loss"]) + 1), v["history"]["val_loss"], ls, c=c, marker="o", label=k)
    ax[1].plot(range(1, len(v["history"]["val_acc"]) + 1), v["history"]["val_acc"], ls, c=c, marker="o", label=k)
ax[1].axhline(BASE["tfidf_logreg"]["accuracy"], c="gray", ls=":", label="TF-IDF + LogReg (test)")
ax[0].set_title("validation loss"); ax[1].set_title("validation accuracy")
for a in ax: a.set_xlabel("epoch"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(IMG / "EC_curves.png", bbox_inches="tight"); plt.show()

### Confusion matrices and ROC curves

Rows = true class, columns = predicted class. The negative class is the hard one: a model that misses negatives is useless to a shop that wants to find unhappy customers.

In [ ]:
fig, ax = plt.subplots(1, len(RUNS) + 1, figsize=(4 * (len(RUNS) + 1), 3.4))
for a, (k, v) in zip(ax, RUNS.items()):
    cm = confusion_matrix(yte, (v["p_test"] >= 0.5).astype(int))
    a.imshow(cm, cmap="Blues")
    for (i, j), n in np.ndenumerate(cm): a.text(j, i, f"{n:,}", ha="center", va="center", color="white" if n > cm.max() / 2 else "black")
    a.set_xticks([0, 1], ["neg", "pos"]); a.set_yticks([0, 1], ["neg", "pos"]); a.set_title(k, fontsize=9)
for k, v in RUNS.items():
    fpr, tpr, _ = roc_curve(yte, v["p_test"]); ax[-1].plot(fpr, tpr, label=f"{k} ({v['roc_auc']:.3f})")
fpr, tpr, _ = roc_curve(yte, p_lr); ax[-1].plot(fpr, tpr, "k:", label=f"TF-IDF ({BASE['tfidf_logreg']['roc_auc']:.3f})")
ax[-1].plot([0, 1], [0, 1], c="#ccc"); ax[-1].set_title("ROC"); ax[-1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(IMG / "EC_confusion_roc.png", bbox_inches="tight"); plt.show()

### Does the LSTM read long reviews better?

Accuracy by review length. If the RNN loses information over many steps (the vanishing gradient of the concepts notebook), its accuracy should drop more on long reviews than the LSTM's. Remember the pre-padding: for a 10-word review the first 90 steps are padding, and the real words come last.

In [ ]:
n_te = df.n_words.values[i_te]
bins = [0, 10, 20, 40, 70, 100, 10_000]; labels = ["1-10", "11-20", "21-40", "41-70", "71-100", ">100 (cut)"]
bucket = pd.cut(n_te, bins, labels=labels)
acc_len = pd.DataFrame({k: pd.Series((v["p_test"] >= 0.5) == yte).groupby(bucket).mean() for k, v in RUNS.items()})
acc_len["reviews"] = pd.Series(bucket).value_counts().reindex(labels).values
display(acc_len.round(3))
acc_len.drop(columns="reviews").plot(marker="o", figsize=(8, 3.6), title="test accuracy by review length (words)")
plt.ylabel("accuracy"); plt.savefig(IMG / "EC_by_length.png", bbox_inches="tight"); plt.show()

### Example predictions

The same test reviews scored by the RNN and the LSTM (Keras runs), including reviews where the two disagree.

In [ ]:
pr, pl = RUNS["EC-TF-RNN"]["p_test"], RUNS["EC-TF-LSTM"]["p_test"]
ex = pd.DataFrame({"review": texts[i_te], "stars": df.Rating.values[i_te], "P(pos) RNN": pr.round(3), "P(pos) LSTM": pl.round(3)})
dis = ex[((pr >= 0.5) != (pl >= 0.5))]
print(f"the two models disagree on {len(dis):,} of {len(ex):,} test reviews")
pd.set_option("display.max_colwidth", 160)
display(pd.concat([ex.head(4), dis.sample(min(6, len(dis)), random_state=1)]))

## 8. Conclusion

Written after the full run (see the table in §7).